- The last layer(s) of the network should be a classifier for sentiment polarity, and should be designed by you. There is no requirement on the technical complexity of this classifier. This implies that you can't just load a ready-made pipeline.
- Use pyTorch's Dataset and DataLoader classes for loading the data.
- Train the network using batches of at least 2.
- Compare performance between some model and a similar distilled one (e.g. BERT and DistilBERT).


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import torchinfo
import transformers

import gc
gc.collect()

if torch.xpu.is_available():
    device = torch.device("xpu:0")
    available, total = torch.xpu.mem_get_info(device)
    print(f"{total / 1024 ** 3:.1f} GB, {100 * available / total:.1f}% tillgängligt.")
elif torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")




11.3 GB, 99.4% tillgängligt.


In [7]:
from sentence_transformers import SentenceTransformer
import torch

model = SentenceTransformer(
    "KaLM-Embedding/KaLM-embedding-multilingual-mini-instruct-v2.5",
    trust_remote_code=True,
    model_kwargs={
        "torch_dtype": torch.bfloat16,
        "attn_implementation": "flash_attention_2",  # Optional
    },
)
model.max_seq_length = 512

sentences = ["This is an example sentence", "Each sentence is converted"]
embeddings = model.encode(
    sentences,
    normalize_embeddings=True,
    batch_size=256,
    show_progress_bar=True,
)
print(embeddings)

ImportError: This modeling file requires the following packages that were not found in your environment: flash_attn. Run `pip install flash_attn`

In [ ]:
class SentimentDecisionBoundariesLayer(nn.Module):                              # assuming good euclidean manifolds from the embedding model
    def __init__(self, embedding_dim):                                          # the dimensionality of the embedding
        super().__init__()
        self.linear = nn.Linear(in_features=embedding_dim, 
                                out_features=3,                                 # how many sentiment labels to get logits for! negative, positive, neutral = 3
                                bias=True,
                                device=device,
                                dtype=torch.float)      
    
    def forward(self, X):
        return self.linear.forward(X)
        
model = SentimentDecisionBoundariesLayer(embedding_dim=896)                     # the nr of dims of KaLM-Embedding-V2.5
model = model.to(device)

logits = model.forward(embeddings)
predictions = F.softmax(logits, dim=-1)

print(predictions)

